# Задача 1 - SFT

In [ ]:
%pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.48.2 datasets==4.2.0 scikit-learn==1.7.2 scipy==1.16.2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import json
import pickle
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from scipy.sparse import hstack
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/task_data')
WORK.mkdir(parents=True, exist_ok=True)

def find_file(name):
    found = list(INPUT.rglob(name)) + list(WORK.rglob(name))
    if found:
        return found[0]
    return None

if find_file('kid_adult.jsonl') is None:
    archives = list(INPUT.rglob('*.zip'))
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK)

KID_PATH = find_file('kid_adult.jsonl')
TEST_PATH = find_file('public_test_style.jsonl')
CLF_PATH = find_file('style_clf.pkl')

In [ ]:
def read_jsonl(path):
    with open(path, encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

train_rows = read_jsonl(KID_PATH)
test_rows = read_jsonl(TEST_PATH)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def make_text(row):
    messages = [
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': row['kid']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

texts = [make_text(row) for row in train_rows]
train_dataset = Dataset.from_dict({'text': texts})

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=['text'],
)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map={'': 0},
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False

In [ ]:
training_args = TrainingArguments(
    output_dir='/kaggle/working/task1_train',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    logging_strategy='no',
    disable_tqdm=True,
    save_strategy='no',
    report_to='none',
    seed=SEED,
    data_seed=SEED,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=collator,
)
trainer.train()
model.save_pretrained('/kaggle/working/task1_sft_adapter')
tokenizer.save_pretrained('/kaggle/working/task1_sft_adapter')

In [ ]:
model.eval()
model.config.use_cache = True
answers = []

for row in test_rows:
    messages = [{'role': 'user', 'content': row['prompt']}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    answers.append(answer)

In [ ]:
with open(CLF_PATH, 'rb') as file:
    style_model = pickle.load(file)

vectorizers = style_model['vecs']
classifier = style_model['clf']
features = hstack([vectorizer.transform(answers) for vectorizer in vectorizers])
simple_column = list(classifier.classes_).index(1)
simple_probabilities = classifier.predict_proba(features)[:, simple_column]
p_simple = float(simple_probabilities.mean())

if p_simple < 0.4:
    interval = 'А'
elif p_simple < 0.7:
    interval = 'Б'
elif p_simple < 0.9:
    interval = 'В'
else:
    interval = 'Г'

result = {'task': 1, 'p_simple': p_simple, 'interval': interval, 'seed': SEED}
with open('/kaggle/working/task1_result.json', 'w', encoding='utf-8') as file:
    json.dump(result, file, ensure_ascii=False, indent=2)
with open('/kaggle/working/task1_answers.jsonl', 'w', encoding='utf-8') as file:
    for row, answer, probability in zip(test_rows, answers, simple_probabilities):
        item = {'prompt': row['prompt'], 'answer': answer, 'p_simple': float(probability)}
        file.write(json.dumps(item, ensure_ascii=False) + '\n')

print(result)